### Imports & Downloads

In [25]:
%pip install uv --quiet
%uv pip install pandas numpy plotly matplotlib
%uv sync

Note: you may need to restart the kernel to use updated packages.


c:\Users\specb\Desktop\School\csc5260\project\Modern-Store-Of-Value\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


Using Python 3.11.15 environment at: C:\Users\specb\Desktop\School\csc5260\project\Modern-Store-Of-Value\.venv
Audited 4 packages in 590ms


Note: you may need to restart the kernel to use updated packages.


Resolved 132 packages in 18ms
Audited 128 packages in 485ms


In [26]:
# Data manipulation tools
import pandas as pd
import datetime

# Visualization tools
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# OS tools
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

### Downloads

In [27]:
stooq_tickers = {
    "Crypto ETFs": [
        "BITW",  # Bitwise 10 Crypto Index
        "IBIT",  # iShares Bitcoin Trust (Replaces BTC-USD)
        "ETHA"   # iShares Ethereum Trust (Replaces ETH-USD)
    ], 
    
    "Individual Stocks": [
        "NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"
    ],
    
    "Sector ETFs": [
        "XLU"    # Utilities Select Sector SPDR Fund
    ],
    
    "Broad Market ETFs": [
        "SPY",   # S&P 500
        "VTI"   # Total US Market (replaces Wilshire 5000)
    ],
    
    "Commodity ETFs (Metals)": [
        "GLD",   # Gold (Baseline)
        "SLV",   # Silver
        "PPLT",  # Platinum
        "PALL"   # Palladium
    ],
    
    "Commodity ETFs (Agriculture)": [
        "WEAT",  # Wheat
        "SOYB",  # Soybeans
        "DBA"    # Broad Agriculture
    ]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [28]:
# -----------------------------
# Flatten tickers + category map
# -----------------------------
category_map = {
    ticker: category
    for category, tickers in stooq_tickers.items()
    for ticker in tickers
}

flat_tickers = list(category_map.keys())


# -----------------------------
# Download data
# -----------------------------
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:

    # Stores valid tickers
    valid_tickers = [
        t for t in flat_tickers
        if processor.has_ticker(t)
    ]

    # Prints tickers that were not valid
    missing = sorted(set(flat_tickers) - set(valid_tickers))
    if missing:
        print("Skipping missing tickers:", missing)

    data = processor.download(
        valid_tickers,
        start=start_date,
        end=end_date,
    )


# -----------------------------
# Attach metadata
# -----------------------------
for ticker, frame in data.items():
    data[ticker] = frame.assign(
        Ticker=ticker,
        Category=category_map[ticker],
    )


# -----------------------------
# Combine dataset
# -----------------------------
combined_data = pd.concat(data.values()).reset_index()

# data['AAPL'].tail()
# data["BITW"]
# combined_data[combined_data['Ticker'] == 'BITW']
print(combined_data)
# combined_data

           Date     Open     High      Low    Close       Volume  OpenInt  \
0    2025-12-31  62.8200  63.1200  56.5600  58.7600    1752680.0        0   
1    2026-01-31  59.9400  66.4800  54.5000  55.6601    2425192.0        0   
2    2026-02-28  51.2878  52.2950  40.6599  43.0400    4596906.0        0   
3    2026-03-26  42.9952  49.4500  42.9952  44.8700    1583267.0        0   
4    2024-01-31  26.4000  26.4100  22.0200  24.3000  207876989.0        0   
...         ...      ...      ...      ...      ...          ...      ...   
1307 2025-11-30  26.6000  26.9400  25.5500  26.4200    3917732.0        0   
1308 2025-12-31  26.3600  26.6666  25.4000  25.5200    4877035.0        0   
1309 2026-01-31  25.5000  26.0500  25.4250  25.6600    5404587.0        0   
1310 2026-02-28  25.5500  26.1500  25.5400  26.0200    5422359.0        0   
1311 2026-03-26  26.0400  27.1429  25.8600  27.1100   37145895.0        0   

     Ticker                      Category  
0      BITW                   C

### Filter Crisis and Normal Periods
- Filters the global data into two new dataframes:
    - Crisis periods
    - Normal periods
    

In [29]:
# Get date range
print(f"Earliest Date: {combined_data['Date'].min().date()}")
print(f"Latest Date:   {combined_data['Date'].max().date()}")

Earliest Date: 2021-01-31
Latest Date:   2026-03-26


From quick research these periods can be identified as 'crisis periods':
- 2022-01-01 → 2022-10-15   (Inflation Bear Market)
- 2023-03-08 → 2023-05-15   (Banking Crisis)
- 2025-04-02 → 2025-04-30   (Tariff Shock)
- 2026-02-15 → 2026-03-26   (Geopolitical Selloff)

In [33]:
# Define crisis periods (hard coded for now)
crisis_periods = [
    {
        "name": "Inflation Bear Market", 
        "start": "2022-01-01", 
        "end": "2022-10-15"
    },
    {
        "name": "Banking Crisis", 
        "start": "2023-03-08", 
        "end": "2023-05-15"
    },
    {
        "name": "Tariff Shock", 
        "start": "2025-04-02", 
        "end": "2025-04-30"
    },
    {
        "name": "Geopolitical Stress", 
        "start": "2026-02-15", 
        "end": "2026-03-26"
    }
]


In [34]:
# Classify a date as a crisis date (T/F)

combined_data["Crisis"] = False

for period in crisis_periods:
    start = pd.to_datetime(period["start"])
    end = pd.to_datetime(period["end"])

    mask = combined_data["Date"].between(start, end)
    combined_data.loc[mask, "Crisis"] = True
    
# print(combined_data.head(5))

In [36]:
# Create crisis and normal dataframes
crisis_data = combined_data[combined_data['Crisis'] == True]

non_crisis_data = combined_data[combined_data['Crisis'] == False]

print(crisis_data[['Date', 'Ticker', 'Crisis']].head(5))
print(non_crisis_data[['Date', 'Ticker', 'Crisis']].head(5))

         Date Ticker  Crisis
2  2026-02-28   BITW    True
3  2026-03-26   BITW    True
19 2025-04-30   IBIT    True
29 2026-02-28   IBIT    True
30 2026-03-26   IBIT    True
        Date Ticker  Crisis
0 2025-12-31   BITW   False
1 2026-01-31   BITW   False
4 2024-01-31   IBIT   False
5 2024-02-29   IBIT   False
6 2024-03-31   IBIT   False


### Calculate Success Metric - Volatility Metrics
- Volatility Metrics: Standard deviation of returns to measure price variation over time

In [6]:
# Create returns column

# Sort by ticker and date
df_sorted = combined_data.sort_values(by=['Ticker', 'Date'])

# Gets return values for each month
df_sorted["Return"] = (df_sorted.groupby('Ticker')['Close'].pct_change())

df_sorted.head(5)

,Date,Open,High,Low,Close,Volume,OpenInt,Ticker,Category,Return
115,2021-01-31,129.977,141.231,123.033,128.454,2.301989e+09,0,AAPL,Individual Stocks,NaN
116,2021-02-28,130.202,134.416,115.421,118.212,1.881786e+09,0,AAPL,Individual Stocks,-0.079733
117,2021-03-31,120.647,125.489,113.288,119.085,2.719075e+09,0,AAPL,Individual Stocks,0.007385
118,2021-04-30,120.558,133.639,119.407,128.161,1.938602e+09,0,AAPL,Individual Stocks,0.076214
119,2021-05-31,128.729,130.705,119.377,121.688,1.753673e+09,0,AAPL,Individual Stocks,-0.050507


In [7]:
# Compute volatility

volatility = (df_sorted.groupby('Ticker')['Return'].std()
    .reset_index(name='Volatility'))

volatility

,Ticker,Volatility
0,AAPL,0.070338
1,AMD,0.163142
2,AMZN,0.089615
3,BITW,0.136531
4,DBA,0.033687
5,ETHA,0.228168
6,GLD,0.049994
7,HD,0.071330
8,IBIT,0.158661
9,JNJ,0.048559


### Additional data formatting for visualizations
- Add ticker names as a column
- Sort by volatility values (low to high -> better to worse)
- Create a filtered dataframe of volatility values better (lower) than gold

In [8]:
# Add ticker names for plotting

ticker_names = {
    "AAPL": "Apple Inc.",
    "AMD": "Advanced Micro Devices, Inc.",
    "AMZN": "Amazon.com, Inc.",
    "BITW": "Bitwise 10 Crypto Index Fund",
    "DBA": "Invesco DB Agriculture Fund",
    "ETHA": "iShares Ethereum Trust ETF",
    "GLD": "SPDR Gold Shares",
    "HD": "The Home Depot, Inc.",
    "IBIT": "iShares Bitcoin Trust ETF",
    "JNJ": "Johnson & Johnson",
    "LOW": "Lowe's Companies, Inc.",
    "MSFT": "Microsoft Corporation",
    "NVDA": "NVIDIA Corporation",
    "PALL": "Aberdeen Physical Palladium Shares ETF",
    "PPLT": "Aberdeen Physical Platinum Shares ETF",
    "SLV": "iShares Silver Trust",
    "SOYB": "Teucrium Soybean Fund",
    "SPY": "SPDR S&P 500 ETF Trust",
    "TSLA": "Tesla, Inc.",
    "VTI": "Vanguard Total Stock Market ETF",
    "WEAT": "Teucrium Wheat Fund",
    "WMT": "Walmart Inc.",
    "XLU": "Utilities Select Sector SPDR Fund"
}

volatility['Name'] = volatility['Ticker'].map(ticker_names)
volatility = volatility[['Ticker', 'Name', 'Volatility']]
volatility

,Ticker,Name,Volatility
0,AAPL,Apple Inc.,0.070338
1,AMD,"Advanced Micro Devices, Inc.",0.163142
2,AMZN,"Amazon.com, Inc.",0.089615
3,BITW,Bitwise 10 Crypto Index Fund,0.136531
4,DBA,Invesco DB Agriculture Fund,0.033687
5,ETHA,iShares Ethereum Trust ETF,0.228168
6,GLD,SPDR Gold Shares,0.049994
7,HD,"The Home Depot, Inc.",0.071330
8,IBIT,iShares Bitcoin Trust ETF,0.158661
9,JNJ,Johnson & Johnson,0.048559


In [9]:
# Sort by volatility

vol_sorted = volatility.sort_values('Volatility').reset_index(drop=True)
vol_sorted


,Ticker,Name,Volatility
0,DBA,Invesco DB Agriculture Fund,0.033687
1,SPY,SPDR S&P 500 ETF Trust,0.043866
2,VTI,Vanguard Total Stock Market ETF,0.044492
3,SOYB,Teucrium Soybean Fund,0.045272
4,JNJ,Johnson & Johnson,0.048559
5,XLU,Utilities Select Sector SPDR Fund,0.049710
6,GLD,SPDR Gold Shares,0.049994
7,WMT,Walmart Inc.,0.056843
8,MSFT,Microsoft Corporation,0.067243
9,WEAT,Teucrium Wheat Fund,0.067703


In [10]:
# Get baseline volatility for gold

gold_vol = vol_sorted[vol_sorted['Ticker'] == 'GLD']['Volatility'].values[0]
# silver_vol = vol_sorted[vol_sorted['Ticker'] == 'SLV']['Volatility'].values[0]
# plat_vol = vol_sorted[vol_sorted['Ticker'] == 'PPLT']['Volatility'].values[0]
# pall_vol = vol_sorted[vol_sorted['Ticker'] == 'PALL']['Volatility'].values[0]

print(gold_vol)

# Filter dataset to contain only tickers with a lower volatility (better) than gold
vol_filtered = vol_sorted[vol_sorted['Volatility'] < gold_vol]

vol_filtered


0.04999374351458432


,Ticker,Name,Volatility
0,DBA,Invesco DB Agriculture Fund,0.033687
1,SPY,SPDR S&P 500 ETF Trust,0.043866
2,VTI,Vanguard Total Stock Market ETF,0.044492
3,SOYB,Teucrium Soybean Fund,0.045272
4,JNJ,Johnson & Johnson,0.048559
5,XLU,Utilities Select Sector SPDR Fund,0.049710


### Visualize Volatilities

All data:

- Bar plot of all volatility values with gold baseline
- Table of all tickers, ticker names, and volatility values

Filtered data:

- Bar plot of filtered tickers
- Table of filtered tickers

In [11]:
# Plot volatilities in barchar with gold as baseline

import plotly.express as px

fig = px.bar(
    vol_sorted,
    x="Ticker",
    y="Volatility",
    # color="above_threshold",
    title="Volatility vs Threshold"
)

fig.add_hline(
    y=gold_vol, 
    annotation_text="Gold volatility (benchmark)",
    annotation_position='top left')

fig.update_layout(
    xaxis=dict(
        categoryorder="array",
        categoryarray=vol_sorted["Ticker"]
    ),
    xaxis_tickangle=-45
)

fig.show()

In [12]:
# Table of volatilities

import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=list(vol_sorted[['Ticker', 'Name', 'Volatility']]),
                fill_color="lightgrey",
                align="left"
            ),
            cells=dict(
                values=[
                    vol_sorted["Ticker"],
                    vol_sorted["Name"],
                    round(vol_sorted["Volatility"], 5)
                ],
                align="left"
            )
        )
    ]
)

fig.update_layout(
    title_text = "Volatility Comparison of Modern Store-of-Value Assets",
    title_x = 0.5,
    title_y = 0.85
)

fig.show()

In [ ]:
# Plot volatilities in barchar with gold as baseline (filtered)

import plotly.express as px

fig = px.bar(
    vol_filtered,
    x="Ticker",
    y="Volatility",
    # color="above_threshold",
    title="Volatility vs Threshold (filtered)"
)

fig.add_hline(y=gold_vol, annotation_text="Gold volatility (benchmark)")

fig.update_layout(
    xaxis=dict(
        categoryorder="array",
        categoryarray=vol_filtered["Ticker"]
    )
)

fig.show()

In [ ]:
# Table of volatilities (filtered)

import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=list(vol_filtered[['Ticker', 'Name', 'Volatility']]),
                fill_color="lightgrey",
                align="left"
            ),
            cells=dict(
                values=[
                    vol_filtered["Ticker"],
                    vol_filtered["Name"],
                    round(vol_filtered["Volatility"], 5)
                ],
                align="left"
            )
        )
    ]
)

fig.update_layout(
    title_text = "Volatility Comparison of Modern Store-of-Value Assets (filtered)",
    title_x = 0.5,
    title_y = 0.85
)

fig.show()

### Normal volatility vs Crisis Volatility
- Compare volatility during periods of crisis and normal periods
- Uses filtered datasets from earlier

### Future ideas:
- compare volatility during crisis periods

- 